In [1]:
from azureml.core import Workspace, Dataset
import os

# 1. Connect to your workspace (v1 syntax)
ws = Workspace(
    subscription_id="ab211f7b-463f-4833-9605-d260e596a35a",
    resource_group="73da10b4-5dff-54e2-db0d-3a1fab882485",
    workspace_name="73da10b45dff54e2db0d3a1fab882485"
)

# 2. Get your datastore
datastore = ws.datastores["73da10b45dff54e2db0d3a1fab882485"]

# 3. Create a dataset reference to your root NIFTI folder
nifti_dataset = Dataset.File.from_files(path=(datastore, 'NIFTI/DATA_REGISTERED_FIXED_LATE'))

# 4. Start the interactive mount
mount_context = nifti_dataset.mount()
mount_context.start()

# 5. Get the actual local OS path
local_mount_path = mount_context.mount_point
print(f"Data successfully mounted at: {local_mount_path}")

# Verify by listing the folders inside the mount
print(os.listdir(local_mount_path)[:5])

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Data successfully mounted at: /tmp/tmp98tay2uq
['TAVI_002', 'TAVI_003', 'TAVI_004', 'TAVI_006', 'TAVI_007']


In [2]:
!pwd

/mnt/batch/tasks/shared/LS_root/mounts/clusters/vmprdwe1-vzhst6/code/Users/pisu.francesco/LEMS-CT-project/notebooks


In [2]:
from pathlib import Path

list((Path(local_mount_path) / "TAVI_002").glob("*"))

[PosixPath('/tmp/tmpou_70efn/TAVI_002/CT_ANGIO.nii.gz'),
 PosixPath('/tmp/tmpou_70efn/TAVI_002/CT_LATE.nii.gz'),
 PosixPath('/tmp/tmpou_70efn/TAVI_002/registration_mask.nii.gz')]

In [6]:
import os
import re
import random
import pandas as pd
from pathlib import Path

# 1. Setup paths and regex
input_dir = Path(local_mount_path) 
valid_patients = []
pattern = re.compile(r"^TAVI_\d+$")

print(f"Scanning {input_dir} for valid patients...")

# 2. Find all valid patient folders that have both image and label
for patient_folder in input_dir.iterdir():
    if patient_folder.is_dir() and pattern.match(patient_folder.name):
        img_path = patient_folder / "CT_LATE.nii.gz"
        lbl_path = patient_folder / "registration_mask.nii.gz"
        
        if img_path.exists() and lbl_path.exists():
            valid_patients.append(patient_folder.name)

# 3. Sort first to guarantee identical behavior across different OS/file systems before shuffling
valid_patients.sort()

# 4. Shuffle with your specific seed to maintain consistency with previous experiments
random.seed(1303)
random.shuffle(valid_patients)

# 6. Assign 5 folds evenly using a modulo operator
num_folds = 5
fold_assignments = []

for i, patient_id in enumerate(valid_patients):
    fold = i % num_folds  # Distributes them as: 0, 1, 2, 3, 4, 0, 1, 2...
    fold_assignments.append({"patient_id": patient_id, "fold": fold})

# 7. Create DataFrame and save it directly to your code directory
df = pd.DataFrame(fold_assignments)
csv_path = "../data/cv_splits.csv"
df.to_csv(csv_path, index=False)

print(f"\nSaved 5-fold CV splits to: {csv_path}")
print("\nFirst 10 rows:")
print(df.head(10))
print("\nFold distribution:")
print(df["fold"].value_counts().sort_index())

Scanning /tmp/tmpou_70efn for valid patients...

Saved 5-fold CV splits to: ../data/cv_splits.csv

First 10 rows:
  patient_id  fold
0   TAVI_149     0
1   TAVI_267     1
2   TAVI_071     2
3   TAVI_076     3
4   TAVI_355     4
5   TAVI_104     0
6   TAVI_073     1
7   TAVI_151     2
8   TAVI_201     3
9   TAVI_277     4

Fold distribution:
fold
0    46
1    45
2    45
3    45
4    45
Name: count, dtype: int64


In [8]:
import pandas as pd
from pathlib import Path

def get_files_from_csv(input_dir, csv_path, fold_idx):
    input_dir = Path(input_dir)
    print(input_dir)
    print(csv_path)
    df = pd.read_csv(csv_path)
    print(df)
    
    train_files = []
    val_files = []
    
    for _, row in df.iterrows():
        patient_id = str(row["patient_id"])
        current_fold = int(row["fold"])
        
        # Dynamically build the absolute path using the Azure mount directory
        img_path = input_dir / patient_id / "CT_LATE.nii.gz"
        lbl_path = input_dir / patient_id / "registration_mask.nii.gz"
        
        if img_path.exists() and lbl_path.exists():
            data_dict = {"image": str(img_path), "label": str(lbl_path)}
            
            # Route to Train or Val based on the fold argument
            if current_fold == fold_idx:
                val_files.append(data_dict)
            else:
                train_files.append(data_dict)
                
    print(f"Loaded {len(train_files)} training scans and {len(val_files)} validation scans for Fold {fold_idx}.")
    return train_files, val_files

train_files, val_files = get_files_from_csv(local_mount_path, csv_path, 0)

/tmp/tmpou_70efn
../data/cv_splits.csv
    patient_id  fold
0     TAVI_149     0
1     TAVI_267     1
2     TAVI_071     2
3     TAVI_076     3
4     TAVI_355     4
..         ...   ...
221   TAVI_257     1
222   TAVI_270     2
223   TAVI_374     3
224   TAVI_101     4
225   TAVI_130     0

[226 rows x 2 columns]
Loaded 180 training scans and 46 validation scans for Fold 0.


In [16]:
train_files

[{'image': '/tmp/tmpou_70efn/TAVI_267/CT_LATE.nii.gz',
  'label': '/tmp/tmpou_70efn/TAVI_267/registration_mask.nii.gz'},
 {'image': '/tmp/tmpou_70efn/TAVI_071/CT_LATE.nii.gz',
  'label': '/tmp/tmpou_70efn/TAVI_071/registration_mask.nii.gz'},
 {'image': '/tmp/tmpou_70efn/TAVI_076/CT_LATE.nii.gz',
  'label': '/tmp/tmpou_70efn/TAVI_076/registration_mask.nii.gz'},
 {'image': '/tmp/tmpou_70efn/TAVI_355/CT_LATE.nii.gz',
  'label': '/tmp/tmpou_70efn/TAVI_355/registration_mask.nii.gz'},
 {'image': '/tmp/tmpou_70efn/TAVI_073/CT_LATE.nii.gz',
  'label': '/tmp/tmpou_70efn/TAVI_073/registration_mask.nii.gz'},
 {'image': '/tmp/tmpou_70efn/TAVI_151/CT_LATE.nii.gz',
  'label': '/tmp/tmpou_70efn/TAVI_151/registration_mask.nii.gz'},
 {'image': '/tmp/tmpou_70efn/TAVI_201/CT_LATE.nii.gz',
  'label': '/tmp/tmpou_70efn/TAVI_201/registration_mask.nii.gz'},
 {'image': '/tmp/tmpou_70efn/TAVI_277/CT_LATE.nii.gz',
  'label': '/tmp/tmpou_70efn/TAVI_277/registration_mask.nii.gz'},
 {'image': '/tmp/tmpou_70efn/TAV

In [11]:
from monai.transforms import (
    EnsureChannelFirstd, Compose,
    CropForegroundd, LoadImaged, Orientationd, RandFlipd,
    RandCropByPosNegLabeld, ScaleIntensityRanged, Spacingd,
    RandRotate90d, ResizeWithPadOrCropd
)

def get_transforms(
    target_spacing: list,
    roi_size: list,
    a_min: float = -100.0,
    a_max: float = 400.0,
    b_min: float = 0.0,
    b_max: float = 1.0,
    flip_prob: float = 0.10,
    rotate_prob: float = 0.10
):
    train_transforms = Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        Spacingd(keys=["image", "label"], pixdim=target_spacing, mode=("bilinear", "nearest")),
        ScaleIntensityRanged(keys=["image"], a_min=a_min, a_max=a_max, b_min=b_min, b_max=b_max, clip=True),
        RandCropByPosNegLabeld(
            keys=["image", "label"], label_key="label", spatial_size=roi_size, 
            pos=1, neg=1, num_samples=4, image_key="image", image_threshold=0, allow_smaller=True
        ),
        ResizeWithPadOrCropd(keys=["image", "label"], spatial_size=roi_size, mode='constant'),
        RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=flip_prob),
        RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=flip_prob),
        RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=flip_prob),
        RandRotate90d(keys=["image", "label"], prob=rotate_prob, max_k=3)
    ])

    val_transforms = Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        Spacingd(keys=["image", "label"], pixdim=target_spacing, mode=("bilinear", "nearest")),
        ScaleIntensityRanged(keys=["image"], a_min=a_min, a_max=a_max, b_min=b_min, b_max=b_max, clip=True),
    ])
    
    return train_transforms, val_transforms

In [13]:
for d in train_files:
    image_path = d["image"]
    label_path = d["label"]
    
# 1. Get the transforms (same as in main)
train_transforms, val_transforms = get_transforms((0.5, 0.5, 0.5),
                                                  (96, 96, 96))   # or however you call it

print("=== Train transforms ===")
for t in train_transforms.transforms:
    print(type(t).__name__)
print("\n=== Val transforms ===")
for t in val_transforms.transforms:
    print(type(t).__name__)
    
        

=== Train transforms ===
LoadImaged
EnsureChannelFirstd
Orientationd
CropForegroundd
Spacingd
ScaleIntensityRanged
RandCropByPosNegLabeld
ResizeWithPadOrCropd
RandFlipd
RandFlipd
RandFlipd
RandRotate90d

=== Val transforms ===
LoadImaged
EnsureChannelFirstd
Orientationd
CropForegroundd
Spacingd
ScaleIntensityRanged


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [33]:
import nibabel as nib
import numpy as np

bad_files = []
empty_masks = []

print(f"Scanning {len(train_files)} training pairs for raw corruption...")

for idx, file_dict in enumerate(train_files):
    img_path = file_dict['image']
    lbl_path = file_dict['label']
    
    try:
        # 1. Can we even read the files?
        img_nii = nib.load(img_path)
        lbl_nii = nib.load(lbl_path)
        
        # 2. Extract the raw numpy arrays
        img_data = img_nii.get_fdata()
        lbl_data = lbl_nii.get_fdata()
        
        # 3. Shape Match Check
        if img_data.shape != lbl_data.shape:
            print(f"\n[SHAPE MISMATCH] {img_path}")
            print(f"  CT Shape:   {img_data.shape}")
            print(f"  Mask Shape: {lbl_data.shape}")
            bad_files.append(file_dict)
            continue
            
        # 4. NaN / Infinity Check (Poison for Neural Networks)
        if np.isnan(img_data).any() or np.isinf(img_data).any():
            print(f"\n[CORRUPT PIXELS] NaNs or Infs detected in: {img_path}")
            bad_files.append(file_dict)
            continue
            
        # 5. Empty Mask Check
        unique_labels = np.unique(lbl_data)
        if len(unique_labels) == 1 and unique_labels[0] == 0:
            print(f"\n[EMPTY MASK WARNING] Mask contains ONLY background (0s): {lbl_path}")
            empty_masks.append(file_dict)
            
    except Exception as e:
        print(f"\n[FATAL READ ERROR] Failed to load: {img_path} or {lbl_path}")
        print(f"  Error: {e}")
        bad_files.append(file_dict)
        
    if (idx + 1) % 20 == 0:
        print(f"Scanned {idx + 1}/{len(train_files)} pairs...")

print("\n--- Diagnostic Results ---")
print(f"Severely Corrupted Files: {len(bad_files)}")
print(f"Empty Masks (Valid but risky): {len(empty_masks)}")

if bad_files:
    print(f"First bad file to investigate: {bad_files[0]['image']}")

Scanning 180 training pairs for raw corruption...
Scanned 20/180 pairs...
Scanned 40/180 pairs...
Scanned 60/180 pairs...
Scanned 80/180 pairs...
Scanned 100/180 pairs...
Scanned 120/180 pairs...
Scanned 140/180 pairs...
Scanned 160/180 pairs...
Scanned 180/180 pairs...

--- Diagnostic Results ---
Severely Corrupted Files: 0
Empty Masks (Valid but risky): 0


# Train

In [ ]:
import os
import re
import random
import torch
from tqdm import tqdm
import numpy as np
from pathlib import Path

from monai.utils import set_determinism
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.data import CacheDataset, DataLoader, decollate_batch

from monai.transforms import (
    AsDiscrete, AsDiscreted, EnsureChannelFirstd, Compose,
    CropForegroundd, LoadImaged, Orientationd, RandFlipd,
    RandCropByPosNegLabeld, ScaleIntensityRanged, Spacingd,
    RandRotate90d, Invertd, SaveImaged, ResizeWithPadOrCropd
)
from monai.networks.nets import UNETR

# ---- 1. DYNAMIC FILE LOADING ----
def get_files(input_dir, train_size=0.9, seed=1303):
    input_dir = Path(input_dir)
    valid_files = []
    pattern = re.compile(r"^TAVI_\d+$")
    
    for patient_folder in input_dir.iterdir():
        if patient_folder.is_dir() and pattern.match(patient_folder.name):
            img_path = patient_folder / "CT_ANGIO.nii.gz"
            lbl_path = patient_folder / "TotalSegmentator" / "CT_ANGIO" / "heartchambers_highres" / "heart_myocardium.nii.gz"
            
            if img_path.exists() and lbl_path.exists():
                valid_files.append({"image": str(img_path), "label": str(lbl_path)})
                
    random.seed(seed)
    random.shuffle(valid_files)
    
    split_idx = int(len(valid_files) * train_size)
    return valid_files[:split_idx], valid_files[split_idx:], valid_files

# ---- 2. TRANSFORMS & FUNCTIONS ----
ROI_SIZE = (128, 128, 128)
TARGET_SPACING = (0.5, 0.5, 0.5)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=TARGET_SPACING, mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-100, a_max=400, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandCropByPosNegLabeld(keys=["image", "label"], label_key="label", spatial_size=ROI_SIZE, pos=3, neg=1, num_samples=4, image_key="image", image_threshold=0, allow_smaller=True),
    ResizeWithPadOrCropd(keys=["image", "label"], spatial_size=ROI_SIZE, mode='constant'),
    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.10),
    RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.10),
    RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.10),
    RandRotate90d(keys=["image", "label"], prob=0.10, max_k=3)
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=TARGET_SPACING, mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-200, a_max=1000, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
])

monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [ ]:
# ---- 3. SETUP PIPELINE ----
# Using the variable from your previous notebook cell directly
input_dfs_dir = Path(local_mount_path)
out_dir = Path("./output/models")
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Reading data directly from notebook mount: {input_dfs_dir}")
train_files, val_files, all_files = get_files(input_dfs_dir, train_size=0.9, seed=1303)
print(f"Found {len(all_files)} valid cases. Train: {len(train_files)} | Val: {len(val_files)}")

Reading data directly from notebook mount: /tmp/tmpf4_3vnwf
Found 449 valid cases. Train: 404 | Val: 45


In [ ]:
# Caching datasets (Reduced cache_num slightly so your VM RAM doesn't crash during interactive dev)
# train_ds = CacheDataset(data=train_files, transform=train_transforms, cache_num=10, cache_rate=1.0, num_workers=4)
# val_ds = CacheDataset(data=val_files, transform=val_transforms, cache_num=2, cache_rate=1.0, num_workers=2) 
from monai.data import Dataset
from monai.networks.nets import SegResNet

# Replace the two CacheDataset lines with this:
train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds = Dataset(data=val_files, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SegResNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    init_filters=16,         # Starts small and doubles each layer
    blocks_down=[1, 2, 2, 4], # Standard depth
    blocks_up=[1, 1, 1],
).to(device)

loss_function = DiceFocalLoss(include_background=False, to_onehot_y=True, softmax=True, gamma=2.0, lambda_dice=0.5, lambda_focal=0.5, alpha=0.75)
torch.backends.cudnn.benchmark = True
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

set_determinism(1303)
max_iterations = 5
eval_num = 2

post_label = AsDiscrete(to_onehot=2)
post_pred = AsDiscrete(argmax=True, to_onehot=2)
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)

# 1. Setup Accumulation
accumulation_steps = 4  # With batch_size=2, this mathematically simulates a batch of 8

global_step = 0
dice_val_best = 0.0

# 2. Ensure optimizer is zeroed out before starting!
optimizer.zero_grad()

# 3. Main Loop
while global_step < max_iterations:
    model.train()
    epoch_iterator = tqdm(train_loader, desc="Training (X / X Steps)", dynamic_ncols=True)
    
    for step_idx, batch in enumerate(epoch_iterator):
        # Move data to GPU cleanly
        x, y = (batch["image"].to(device), batch["label"].to(device))
        
        # Forward pass
        logit_map = model(x)
        
        # Calculate loss and scale it down for accumulation
        loss = loss_function(logit_map, y)
        loss = loss / accumulation_steps
        
        # Backward pass (accumulates the gradients invisibly)
        loss.backward()
        
        # Only step the optimizer once we hit our accumulation target
        if ((step_idx + 1) % accumulation_steps == 0) or (step_idx + 1 == len(train_loader)):
            optimizer.step()
            optimizer.zero_grad()
            
        # Scale the displayed loss back up so it looks normal in the progress bar
        display_loss = loss.item() * accumulation_steps
        epoch_iterator.set_description(f"Training ({global_step} / {max_iterations} Steps) (loss={display_loss:2.5f})")
        
        # ---- VALIDATION BLOCK ----
        if (global_step % eval_num == 0 and global_step != 0) or global_step == max_iterations:
            model.eval()
            with torch.no_grad():
                for val_batch in val_loader:
                    val_inputs, val_labels = (val_batch["image"].to(device), val_batch["label"].to(device))
                    
                    # Sliding window inference for validation
                    val_outputs = sliding_window_inference(val_inputs, ROI_SIZE, 4, model, overlap=0.8)
                    
                    # Decollate and apply discrete transforms
                    val_labels_list = decollate_batch(val_labels)
                    val_labels_convert = [post_label(val_label_tensor) for val_label_tensor in val_labels_list]
                    
                    val_outputs_list = decollate_batch(val_outputs)
                    val_output_convert = [post_pred(val_pred_tensor) for val_pred_tensor in val_outputs_list]
                    
                    # Compute metric for this batch
                    dice_metric(y_pred=val_output_convert, y=val_labels_convert)
                
                # Aggregate metric across entire validation set
                dice_val = dice_metric.aggregate().item()
                dice_metric.reset()
                
                if dice_val > dice_val_best:
                    dice_val_best = dice_val
                    torch.save(model.state_dict(), os.path.join(out_dir, "best_metric_model.pth"))
                    print(f"\nModel Saved! New Best Dice: {dice_val_best:.4f}")
                else:
                    print(f"\nValidation complete. Current Dice: {dice_val:.4f} (Best: {dice_val_best:.4f})")
            
            # Switch back to train mode after validation finishes
            model.train()
            
        global_step += 1
        if global_step >= max_iterations:
            break

Training (X / X Steps):   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  | 0/202 [01:03<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 15.57 GiB of which 1.11 GiB is free. Including non-PyTorch memory, this process has 14.46 GiB memory in use. Of the allocated memory 14.16 GiB is allocated by PyTorch, and 152.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch
import gc

# 1. Delete the massive variables from the current namespace
for var in ['x', 'y', 'logit_map', 'loss', 'model']:
    if var in globals():
        del globals()[var]

# 2. Force Python to clean up the unreferenced objects
gc.collect()

# 3. Tell PyTorch to release the reserved VRAM back to the OS
torch.cuda.empty_cache()

print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Allocated: 14.10 GB
Reserved:  14.25 GB


# Inference

In [7]:
import os
import re
import random
import torch
import numpy as np
from pathlib import Path
from monai.networks.nets import SegResNet
from monai.inferers import sliding_window_inference
from monai.data import Dataset, DataLoader, decollate_batch, MetaTensor
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    CropForegroundd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    Invertd,
    AsDiscreted,
    SaveImaged
)

# --- 1. Fetch Unseen Data using your dynamic mount ---
def get_unseen_file(input_dir, seed=1303):
    input_dir = Path(input_dir)
    valid_files = []
    pattern = re.compile(r"^TAVI_\d+$")
    
    for patient_folder in input_dir.iterdir():
        if patient_folder.is_dir() and pattern.match(patient_folder.name):
            img_path = patient_folder / "CT_ANGIO.nii.gz"
            lbl_path = patient_folder / "TotalSegmentator" / "CT_ANGIO" / "heartchambers_highres" / "heart_myocardium.nii.gz"
            
            if img_path.exists() and lbl_path.exists():
                valid_files.append({"image": str(img_path), "label": str(lbl_path)})
                
    random.seed(seed)
    random.shuffle(valid_files)
    
    if len(valid_files) > 200:
        unseen_files = valid_files[200:]
        print(f"Found {len(unseen_files)} unseen patients in the datastore.")
        return unseen_files[0] 
    else:
        raise ValueError("Dataset has 200 or fewer files. No unseen data available!")

# --- 2. Setup Paths, Hardware, & Constants ---
data_dir = local_mount_path 
model_path = Path.cwd().parent.parent / "output_model/named-outputs/output_model/best_metric_model.pth"
output_dir = "./inference_results"

os.makedirs(output_dir, exist_ok=True)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# MUST match training setup
ROI_SIZE = (96, 96, 96) 
TARGET_SPACING = (0.5, 0.5, 0.5) 

# --- 3. Load Model & Strip DDP Prefix ---
print("Loading model architecture...")
model = SegResNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    init_filters=16,         
    blocks_down=[1, 2, 2, 4], 
    blocks_up=[1, 1, 1],
).to(device)

print(f"Loading weights from {model_path}...")
checkpoint = torch.load(model_path, map_location=device)
state_dict = checkpoint["model_state_dict"]

clean_state_dict = {}
for k, v in state_dict.items():
    name = k[7:] if k.startswith("module.") else k
    clean_state_dict[name] = v
    
model.load_state_dict(clean_state_dict)
model.eval()

# --- 4. Pipeline Transforms (Pre & Post) ---
test_file = get_unseen_file(data_dir)
print(f"\nRunning inference on: {test_file['image']}")

# Pre-processing: Exactly matching the physical geometry and signal intensity of train
infer_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    
    # 1. Crops the air around the patient's body (using the CT scan itself)
    CropForegroundd(keys=["image"], source_key="image"),
    
    # 2. Resamples the physical voxel geometry
    Spacingd(keys=["image"], pixdim=TARGET_SPACING, mode=("bilinear")),
    
    # 3. Normalizes the Hounsfield Units into the [0, 1] range the model understands
    ScaleIntensityRanged(keys=["image"], a_min=-100, a_max=400, b_min=0.0, b_max=1.0, clip=True)
])

# Post-processing: Invert spacing/orientation, discretize, and save
post_transforms = Compose([
    Invertd(
        keys="pred",
        transform=infer_transforms,
        orig_keys="image",
        meta_keys="pred_meta_dict",
        orig_meta_keys="image_meta_dict",
        meta_key_postfix="meta_dict",
        nearest_interp=False,
        to_tensor=True,
    ),
    AsDiscreted(keys="pred", argmax=True),
    SaveImaged(
        keys="pred", 
        meta_keys="pred_meta_dict", 
        output_dir=output_dir, 
        output_postfix="pred_mask", 
        output_ext=".nii.gz", 
        resample=False,
        writer="ITKWriter"
    )
])

ds = Dataset(data=[test_file], transform=infer_transforms)
loader = DataLoader(ds, batch_size=1)

# --- 5. Run Inference & Decollate ---
print("Starting sliding window inference...")
with torch.no_grad():
    for batch in loader:
        inputs = batch["image"].to(device)
        
        with torch.cuda.amp.autocast():
            # Get raw logits from the model
            predictions = sliding_window_inference(inputs, ROI_SIZE, 4, model, overlap=0.8)
            
        # Wrap predictions in MetaTensor to carry original image metadata
        src_meta = batch["image"].meta if hasattr(batch["image"], "meta") else batch["image_meta_dict"]
        predictions = MetaTensor(predictions, meta=src_meta)
        
        # Package for post-processing
        batch_data = {
            "image": batch["image"],
            "pred": predictions,
        }
        
        # Pass through Invertd and SaveImaged
        _ = [post_transforms(i) for i in decollate_batch(batch_data)]

print(f"\nDone! Inference successfully inverted and saved to: {output_dir}")

Loading model architecture...
Loading weights from /mnt/batch/tasks/shared/LS_root/mounts/clusters/vmprdwe1-gpu-vzhst6/code/Users/pisu.francesco/output_model/named-outputs/output_model/best_metric_model.pth...


You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Found 249 unseen patients in the datastore.

Running inference on: /tmp/tmp3sucq4c8/TAVI_694/CT_ANGIO.nii.gz
Starting sliding window inference...


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


2026-03-20 12:10:42,375 INFO image_writer.py:197 - writing: inference_results/CT_ANGIO/CT_ANGIO_pred_mask.nii.gz

Done! Inference successfully inverted and saved to: ./inference_results


In [4]:
!cp /tmp/tmp3sucq4c8/TAVI_694/CT_ANGIO.nii.gz ./inference_results/CT_ANGIO_TAVI_694.nii.gz